# Packer 2019 Atlas — Exploratory Analysis

This notebook walks through the Packer 2019 C. elegans scRNA-seq dataset
in the context of the wiring-atlas pipeline.  It is designed to be the
first document a new collaborator reads to understand the data.

**Dataset**: Packer et al. 2019, *Science*  
**Download**: https://data.caltech.edu/records/b1kj4-nh475  
**Pipeline repo**: packer-wiring-atlas


In [ ]:
import sys, os, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import scipy.sparse as sp

sys.path.insert(0, os.path.abspath('..'))
from packer.loader import PackerDataset
from packer.annotation import NeuronMapper, SENSORY_PREFIXES, MOTOR_PREFIXES, INTERNEURON_PREFIXES
from packer.expression import ExpressionMatrix, MARKER_GENES
from packer.genes import GeneSelector

# ── Paths — edit as needed ────────────────────────────────────────
PACKER_PATH  = r'../data/packer2019.h5ad'
WITVLIET_DIR = r'../data/witvliet'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 120,
})

---
## Section 1 — Dataset Overview

How many cells are there?  What tissues are represented?  
What does the embryo time distribution look like?

In [ ]:
ds = PackerDataset(PACKER_PATH)
summary = ds.validate()

print(f'Total cells:         {summary.n_cells:>8,}')
print(f'Total genes:         {summary.n_genes:>8,}')
print(f'Neuron cells:        {summary.n_neuron_cells:>8,}')
print(f'Embryo time range:   {summary.embryo_time_range[0]:.0f} – {summary.embryo_time_range[1]:.0f} min p.f.c.')
print(f'Annotation column:   {summary.annotation_column}')
print(f'Time column:         {summary.time_column}')

In [ ]:
obs = ds.adata.obs
tissue_counts = obs['cell_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: cell type bar chart
top_types = tissue_counts.head(12)
axes[0].barh(top_types.index[::-1], top_types.values[::-1], color='#60A5FA')
axes[0].set_xlabel('Cell count')
axes[0].set_title('Top 12 cell types (Packer 2019)', fontweight='bold')

# Right: embryo time histogram
times = obs['embryo_time'].astype(float)
axes[1].hist(times[times > 0], bins=60, color='#4ADE80', edgecolor='white', lw=0.4)
axes[1].axvline(400, color='#EF4444', ls='--', lw=1.5, label='400 min (approx. hatching)')
axes[1].set_xlabel('Embryo time (min p.f.c.)')
axes[1].set_ylabel('Cell count')
axes[1].set_title('Embryo time distribution', fontweight='bold')
axes[1].legend()

fig.tight_layout()
plt.show()

print(f'\nCells with time > 400 min: {(times > 400).sum():,} ({100*(times > 400).mean():.1f}%)')

---
## Section 2 — Neuron Coverage

Which Witvliet neurons map cleanly to Packer?  
Which rules resolved which neurons?  
The bilateral symmetry assumption is made explicit here.

In [ ]:
# Load Witvliet neuron list
all_n = set()
for st in range(1, 9):
    try:
        with open(os.path.join(WITVLIET_DIR, f'Dataset{st}_synapses.json')) as f:
            d = json.load(f)
        for r in d:
            all_n.add(r['pre'])
            post = r['post'] if isinstance(r['post'], list) else [r['post']]
            all_n.update(post)
    except FileNotFoundError:
        pass

witvliet = sorted(all_n)
print(f'Witvliet neurons: {len(witvliet)}')

In [ ]:
packer_subs = obs[ds.annotation_column].dropna().unique().tolist()
mapper = NeuronMapper(packer_subs)
report = mapper.report(witvliet)

print(f'Mapped:   {report.n_neurons_mapped}/{report.n_neurons_total} ({100*report.coverage_fraction:.1f}%)')
print(f'Unmapped: {report.n_neurons_unmapped}')
print()

# Rule breakdown table
rule_df = pd.DataFrame([
    {'Rule': r, 'Description': report.rule_descriptions[r], 'Count': len(v)}
    for r, v in report.by_rule.items() if v
])
print(rule_df.to_string(index=False))

In [ ]:
# Bilateral expansion: show subtypes where >1 Witvliet name maps to same Packer class
bilateral = {
    k: v for k, v in report.bilateral_pairs.items() if len(v) >= 2
}
bilateral_df = pd.DataFrame([
    {'Packer subtype': k, 'Witvliet names': ', '.join(v), 'N neurons': len(v)}
    for k, v in sorted(bilateral.items(), key=lambda x: -len(x[1]))
]).head(20)

print('Bilateral pairs (first 20):')
print(bilateral_df.to_string(index=False))
print(f'\nTotal bilateral packer subtypes: {len(bilateral)}')
print('Limitation: L/R neurons get identical expression; the model cannot distinguish them.')

---
## Section 3 — Embryo Time by Neuron Type

Different neuron classes complete terminal differentiation at different times.  
This shows why a global time cutoff is inferior to a per-subtype percentile.

In [ ]:
FOCUS_TYPES = ['AWC', 'DA', 'AVA', 'AIY', 'RIA', 'ASE', 'AIA']

fig, axes = plt.subplots(1, len(FOCUS_TYPES), figsize=(16, 3.5), sharey=False)
colors = plt.cm.Set2(np.linspace(0, 1, len(FOCUS_TYPES)))

nv = ds.neuron_view()
nv_obs = nv.obs

for ax, ntype, color in zip(axes, FOCUS_TYPES, colors):
    mask = nv_obs[ds.annotation_column] == ntype
    times = nv_obs[ds.time_column].astype(float)[mask]
    if len(times) == 0:
        ax.set_title(f'{ntype}\n(no cells)')
        ax.axis('off')
        continue
    p75 = np.percentile(times, 75)
    ax.hist(times, bins=20, color=color, edgecolor='white', lw=0.3)
    ax.axvline(p75, color='red', ls='--', lw=1.5)
    ax.set_title(f'{ntype}\n(n={len(times)}, p75={p75:.0f})', fontsize=9, fontweight='bold')
    ax.set_xlabel('min p.f.c.', fontsize=8)

fig.suptitle('Embryo time distribution by neuron type  (red = 75th percentile / terminal diff. threshold)',
             fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

---
## Section 4 — Expression Profiles

Heatmap of five neuron types × the curated guidance gene subset.  
Each row is the mean expression at terminal differentiation.  
The pattern of cell-type specificity is the signal the GNN exploits.

In [ ]:
mapping = mapper.build_mapping(witvliet)
gene_id_map = ds.gene_name_map()
gene_name_to_id = ds.gene_id_map()
selector = GeneSelector(gene_id_map)

# Find curated gene IDs that exist in Packer
from packer.genes import _REGISTRY
curated_ids = [gene_name_to_id[n] for n, _, _ in _REGISTRY if n in gene_name_to_id]
curated_names = [n for n, _, _ in _REGISTRY if n in gene_name_to_id]

expr_builder = ExpressionMatrix(
    dataset=nv, mapping=mapping,
    ann_col=ds.annotation_column, time_col=ds.time_column,
    gene_id_to_name=gene_id_map
)

FOCUS_NEURONS = ['AWCL', 'AWCR', 'AVAL', 'AVAR', 'AIYL', 'AIYR', 'ASEL', 'ASER', 'DA1', 'DA2']
result = expr_builder.build(target_neurons=FOCUS_NEURONS, gene_ids=curated_ids)

# Z-score across neurons (column-wise)
X = result.X.copy()
col_std = X.std(0)
col_std[col_std < 1e-9] = 1.0
X_z = (X - X.mean(0)) / col_std
X_z = np.clip(X_z, -3, 3)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(X_z, aspect='auto', cmap='RdBu_r', vmin=-3, vmax=3)
ax.set_xticks(range(len(result.gene_names)))
ax.set_xticklabels(result.gene_names, rotation=60, ha='right', fontsize=8, fontstyle='italic')
ax.set_yticks(range(len(result.neuron_names)))
ax.set_yticklabels(result.neuron_names, fontsize=9)
ax.set_title('Expression Profiles: 10 neurons × curated guidance genes  (Z-scored)', fontweight='bold')
fig.colorbar(im, ax=ax, shrink=0.8, label='Z-score')
fig.tight_layout()
plt.show()

print('Rows with zero expression (unmapped):', result.unmapped_neurons)

---
## Section 5 — Marker Gene Validation

For each known marker, compare expression in its expected neuron type
versus all other neurons.  A fold change > 2 is consistent with correct
cell-type resolution in the expression profiles.

In [ ]:
# Build expression for ALL Witvliet neurons with all genes
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    all_result = expr_builder.build(target_neurons=witvliet)

checks = expr_builder.check_markers(all_result)

fig, axes = plt.subplots(2, 5, figsize=(15, 5))
axes = axes.flat

for ax, m in zip(axes, checks):
    if not m.gene_found:
        ax.text(0.5, 0.5, f'{m.marker_gene}\nnot in dataset',
                ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
        continue

    bars = ax.bar(['Target\n' + m.neuron_type, 'All others'],
                  [m.expression_in_type, m.expression_in_others],
                  color=['#4ADE80' if m.fold_change >= 2 else '#EF4444', '#94A3B8'])
    ax.set_title(f'{m.marker_gene} → {m.neuron_type}\nFC={m.fold_change:.2f}', fontsize=9, fontweight='bold')
    ax.set_ylabel('Mean expr.', fontsize=8)

fig.suptitle('Marker Gene Validation  (green bars = fold change ≥ 2  ✓)', fontweight='bold')
fig.tight_layout()
plt.show()

---
## Section 6 — GNN Coverage Summary

How many Witvliet neurons get real vs zero expression vectors?  
The motor neuron gap is the primary limitation of the current pipeline.

In [ ]:
def circuit_class(name):
    for p in SENSORY_PREFIXES:
        if name.startswith(p): return 'sensory'
    for p in MOTOR_PREFIXES:
        if name.startswith(p): return 'motor'
    for p in INTERNEURON_PREFIXES:
        if name.startswith(p): return 'interneuron'
    return 'other'

mapping = mapper.build_mapping(witvliet)
rows = []
for n in witvliet:
    rows.append({
        'neuron': n,
        'circuit': circuit_class(n),
        'mapped': n in mapping,
        'packer_subtype': mapping.get(n, '—'),
    })
df = pd.DataFrame(rows)

# Summary table
summary_df = df.groupby('circuit').agg(
    total=('neuron', 'count'),
    covered=('mapped', 'sum'),
).assign(
    missing=lambda d: d['total'] - d['covered'],
    coverage_pct=lambda d: (100 * d['covered'] / d['total']).round(1)
)
print('Coverage by circuit module:')
print(summary_df.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(summary_df))
w = 0.35
ax.bar(x - w/2, summary_df['covered'], w, label='Covered (real expression)', color='#4ADE80')
ax.bar(x + w/2, summary_df['missing'], w, label='Missing (zero vector)', color='#F87171')
ax.set_xticks(x)
ax.set_xticklabels(summary_df.index, fontweight='bold')
ax.set_ylabel('Neuron count')
ax.set_title('GNN Feature Coverage by Circuit Module', fontweight='bold')
ax.legend()
for i, (idx, row) in enumerate(summary_df.iterrows()):
    ax.text(i - w/2, row['covered'] + 0.5, f"{row['coverage_pct']}%", ha='center', fontsize=9)
fig.tight_layout()
plt.show()

print('\nMotor neuron note: VB1 and VB2 are the only motor neurons in Witvliet D1.')
print('Neither has a Packer subtype match — this is a known dataset coverage gap.')